In [7]:
## charger le fichier brut

import pandas as pd
from pathlib import Path

projet = Path(r"C:\Projets\permis-quebec")
fichier = list((projet / "data" / "raw").glob("*.csv"))[0]

df = pd.read_csv(fichier, low_memory=False)
print("Lignes au départ :", len(df))

Lignes au départ : 71612


In [8]:
## retirer les doublons
avant = len(df)
df = df.drop_duplicates()
print("Doublons retirés :", avant - len(df))
print("Lignes restantes :", len(df))

Doublons retirés : 2303
Lignes restantes : 69309


In [9]:
## convertir la date et créer l’année et le mois

df["DATE_DELIVRANCE"] = pd.to_datetime(df["DATE_DELIVRANCE"], errors="coerce")
print("Dates invalides :", df["DATE_DELIVRANCE"].isna().sum())

df["annee"] = df["DATE_DELIVRANCE"].dt.year
df["mois"] = df["DATE_DELIVRANCE"].dt.strftime("%Y-%m")

Dates invalides : 0


In [10]:
## nettoyer le texte et regrouper les territoires voisins
for colonne in ["TYPE_PERMIS", "DOMAINE", "RAISON", "ARRONDISSEMENT"]:
    df[colonne] = df[colonne].str.strip()

hors = df["ARRONDISSEMENT"].str.contains("Saint-Augustin|Ancienne-Lorette|Wendake")
print("Lignes hors arrondissements :", hors.sum())
df.loc[hors, "ARRONDISSEMENT"] = "Hors arrondissements"

df["TYPE_PERMIS"] = df["TYPE_PERMIS"].fillna("Type non précisé")


Lignes hors arrondissements : 25


In [5]:
## le contrôle croisé avec SQL
print("Lignes :", len(df))
print("Permis distincts :", df["NUMERO_PERMIS"].nunique())
print(df["TYPE_PERMIS"].value_counts())

Lignes : 69309
Permis distincts : 69306
TYPE_PERMIS
Certificat d'autorisation         42676
Permis de construction            24010
Permis de lotissement              2479
Lettre/ Document administratif      136
Type non précisé                      8
Name: count, dtype: int64


In [6]:
## enregistrer la table nettoyée et noter les décisions

sortie = projet / "data" / "clean" / "permis_clean.csv"
df.to_csv(sortie, index=False, encoding="utf-8-sig")
print("Fichier enregistré :", sortie, "-", len(df), "lignes")

Fichier enregistré : C:\Projets\permis-quebec\data\clean\permis_clean.csv - 69309 lignes


## Décisions de nettoyage

| Décision | Lignes touchées | Justification |
|---|---|---|
| Retrait des doublons exacts | 2 303 | Lignes identiques sur toutes les colonnes |
| Conversion de la date | 0 invalide | Format AAAA-MM-JJ déjà propre |
| Regroupement « Hors arrondissements » | 25 | Saint-Augustin, L'Ancienne-Lorette et Wendake ne sont pas des arrondissements |
| Type manquant remplacé par « Type non précisé » | 8 | Conservés pour ne pas fausser le total |
| Permis sans numéro | 3 | Conservés : les autres colonnes sont renseignées |

Résultat : 69 309 lignes, 69 306 permis distincts. Chiffres identiques à ceux obtenus en SQL.